In [32]:
import zipfile
from itertools import chain
from collections import defaultdict

import pandas as pd

In [33]:
data_path           = '../data'
join_lb_gt_path     = f'{data_path}/lakebench_gt/opendata_join_ground_truth.csv'
union_lb_gt_path    = f'{data_path}/lakebench_gt/opendata_union_ground_truth.csv'

## How many perfect duplicated tables there are into each dataset?

Some tables seems to be actually a duplicate

In [ ]:
from tqdm import tqdm
import concurrent.futures

def process_country(country, data_path):
    country_hash_counts = defaultdict(int)
    with zipfile.ZipFile(f'{data_path}/datasets/datasets_{country}.zip') as z:
        for fname in tqdm(z.namelist(), leave=False):
            with z.open(f'{fname}') as fr:
                country_hash_counts[hash(fr.read())] += 1
    return country, country_hash_counts

countries = ['SG', 'USA', 'UK', 'CAN']

hash_counts = defaultdict(lambda: defaultdict(int))
def f(country):
    return process_country(country, data_path)

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = executor.map(f, countries)

    for country, country_hash_counts in results:
        hash_counts[country].update(country_hash_counts)

defaultdict(<function <lambda> at 0x7f3b1129d080>, {'SG': defaultdict(<class 'int'>, {0: 1, -4674875556263080793: 1, -7634741486389042413: 1, 4263132111951713335: 1, -6302499288434989255: 1, 4324209699969771385: 1, -6435633116165740761: 2, -651822304665593260: 2, 1279420588976299611: 2, 3736502434966160822: 1, 4396213993875151283: 1, 2475886754674689859: 1, -5352036375098766840: 1, 5324245222568634465: 1, -2028616219097356053: 1, 5359489003082805105: 1, -210506810777125347: 1, -1971028898206908090: 1, 5037293001939319807: 1, -6047190395788212743: 1, 7668045894178019434: 1, 6699206777213831161: 1, -456163232713596945: 1, 8285159829220846430: 1, -2324800779446475024: 1, 6509605649039986618: 1, 19750657508789322: 1, 3731304696159723041: 1, -4305375258353556421: 1, -8623446016486027029: 1, 5883705141412307463: 1, -7745546256398028881: 1, 7823516721347420741: 1, -4004241964684670027: 1, -4647055146104747873: 1, 6171706816502946508: 1, 5748786110265944514: 1, 3032368955369073361: 1, 83916144

In [31]:
for country, hashes in hash_counts.items():
    non_unique = len([h for h, hc in hashes.items() if hc > 1])
    print(f'{country=}, {len(hashes)=}, {non_unique=} ({non_unique * 100 // len(hashes)}%)')

country='SG', len(hashes)=1233, non_unique=24 (1%)
country='USA', len(hashes)=3593, non_unique=388 (10%)
country='UK', len(hashes)=1094, non_unique=12 (1%)
country='CAN', len(hashes)=3746, non_unique=226 (6%)


## JOIN

In [3]:
lb_join_gt = pd.read_csv(join_lb_gt_path)
lb_join_gt

,query_table,candidate_table,query_column,candidate_column
0,CAN_CSV0000000000001724__13.csv,CAN_CSV0000000000001787__9.csv,RentLocation,RentLocation
1,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,Post-Op Q Anxiety,Post-Op Q Anxiety
2,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,Post-Op Q Wound,Post-Op Q Wound
3,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,STD_VOLUME_GROUP,STD_VOLUME_GROUP
4,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,ROUTE_SIGNING,ROUTE_SIGNING
...,...,...,...,...
42558,CAN_CSV0000000000013788.csv,CAN_CSV0000000000001071.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42559,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000660.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42560,CAN_CSV0000000000013788.csv,CAN_CSV0000000000013712.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""
42561,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000857.csv,"ï»¿""REF_DATE""","ï»¿""REF_DATE"""


### Are present all the tables from the Ground Truth?

Answer: YES, all the tables listed into the JOIN Ground Truth are also contained into the relative country dataset

In [4]:
all_join_table_ids = set(filter(lambda s: '__' not in s, chain(*lb_join_gt.values.tolist())))
len(all_join_table_ids)

2508

In [11]:
join_table_ids_by_country = {
    country: {s for s in all_join_table_ids if country in s}
    for country in ['SG', 'USA', 'UK', 'CAN']
}

In [12]:
for country, tables in join_table_ids_by_country.items():
    print(f'{country=}, {len(tables)=}')

country='SG', len(tables)=50
country='USA', len(tables)=334
country='UK', len(tables)=103
country='CAN', len(tables)=961


In [13]:
for country, tables in join_table_ids_by_country.items():
    with zipfile.ZipFile(f'{data_path}/datasets/datasets_{country}.zip') as z:
        names = {f.removeprefix(f'datasets_{country}/') for f in z.namelist()}
        names.remove('')

        diff = tables.difference(names)
        print(f'{country=}, {len(diff)=}')

country='SG', len(diff)=0
country='USA', len(diff)=0
country='UK', len(diff)=0
country='CAN', len(diff)=0


### How many Ground Truth pairs have different names for query and candidate columns?

Answer: None

In [8]:
lb_join_gt[lb_join_gt['query_column'] != lb_join_gt['candidate_column']]

,query_table,candidate_table,query_column,candidate_column


### There are some identical query and candidate tables?

Answer: the 4% of the total JOIN pairs into the Ground Truth is actually a no-sense join on the same column of the same table

In [9]:
lb_join_gt[lb_join_gt['query_table'] == lb_join_gt['candidate_table']].shape[0] * 100 / lb_join_gt.shape[0]

4.010525573855227

In [10]:
lb_join_gt[(lb_join_gt['query_table'] == lb_join_gt['candidate_table']) & (lb_join_gt['query_column'] != lb_join_gt['candidate_column'])]

,query_table,candidate_table,query_column,candidate_column


## UNION

In [12]:
lb_union_gt = pd.read_csv(union_lb_gt_path)
lb_union_gt

,query_table,candidate_table
0,CAN_CSV0000000000000474.csv,CAN_CSV0000000000004972.csv
1,CAN_CSV0000000000000474.csv,CAN_CSV0000000000013001.csv
2,CAN_CSV0000000000000474.csv,CAN_CSV0000000000002292.csv
3,CAN_CSV0000000000000474.csv,CAN_CSV0000000000027218.csv
4,CAN_CSV0000000000000474.csv,CAN_CSV0000000000005832.csv
...,...,...
49510,USA_CSV0000000000037302.csv,USA_CSV0000000000037302.csv
49511,USA_CSV0000000000037322.csv,USA_CSV0000000000037327.csv
49512,USA_CSV0000000000037322.csv,USA_CSV0000000000037322.csv
49513,USA_CSV0000000000037327.csv,USA_CSV0000000000037327.csv


### There are pairs of identical tables?

Answer: YES, the 5% (2688) pairs of tables from the Ground Truth is no-sense, since it is references the same identical table

In [13]:
lb_union_gt[lb_union_gt['query_table'] == lb_union_gt['candidate_table']].shape[0] * 100 / lb_union_gt.shape[0]

5.428657982429566

### All the tables are present into the provided dataset?

Filtering the slices

Answer: YES, it seems that all the tables in the UNION Ground Truth are present into the relative country Open Data

In [14]:
all_union_table_ids = set(filter(lambda s: '__' not in s, chain(*lb_union_gt.values.tolist())))
len(all_union_table_ids)

3784

In [15]:
union_table_ids_by_country = {
    country: {s for s in all_union_table_ids if country in s}
    for country in ['SG', 'USA', 'UK', 'CAN']
}

In [16]:
for country, tables in union_table_ids_by_country.items():
    print(f'{country=}, {len(tables)=}')

country='SG', len(tables)=355
country='USA', len(tables)=700
country='UK', len(tables)=234
country='CAN', len(tables)=2495


In [17]:
for country, tables in union_table_ids_by_country.items():
    with zipfile.ZipFile(f'{data_path}/datasets/datasets_{country}.zip') as z:
        names = {f.removeprefix(f'datasets_{country}/') for f in z.namelist()}
        names.remove('')

        diff = tables.difference(names)
        print(f'{country=}, {len(diff)=}')


country='SG', len(diff)=0
country='USA', len(diff)=0
country='UK', len(diff)=0
country='CAN', len(diff)=0
